In [0]:
# ============================================================
# CELL 1 — Setup: Create folders using Unity Catalog Volumes
# ============================================================
# Unity Catalog is Databricks' modern storage system.
# It's more secure than DBFS and is what banks actually use.
# We use spark.sql to create our catalog structure.

# Step 1 — Create a catalog for our project
spark.sql("CREATE CATALOG IF NOT EXISTS aml_pipeline")

# Step 2 — Create a database/schema inside the catalog
spark.sql("CREATE DATABASE IF NOT EXISTS aml_pipeline.transactions")

# Step 3 — Create a Volume (like a folder for raw files)
spark.sql("CREATE VOLUME IF NOT EXISTS aml_pipeline.transactions.raw_data")

print("✅ Unity Catalog structure created!")
print("📁 Catalog:  aml_pipeline")
print("📁 Database: aml_pipeline.transactions")
print("📁 Volume:   aml_pipeline.transactions.raw_data")

✅ Unity Catalog structure created!
📁 Catalog:  aml_pipeline
📁 Database: aml_pipeline.transactions
📁 Volume:   aml_pipeline.transactions.raw_data


In [0]:
# ============================================================
# CELL 2 — Generate sample transactions into our Volume
# ============================================================
# This writes 100 fake transactions as a JSON file into our
# Unity Catalog Volume. Think of it as simulating your Mac's
# transaction generator uploading a batch of transactions.

import json
import random
import uuid
from datetime import datetime, timezone

# -- Sample data lists ---------------------------------------
SANCTIONED_NAMES = ["Viktor Bout", "Semion Mogilevich", "Ramzan Kadyrov"]
HIGH_RISK        = ["KP", "IR", "RU", "SY"]
NORMAL_COUNTRIES = ["US", "GB", "DE", "FR", "JP", "CA", "AU"]
CURRENCIES       = ["USD", "EUR", "GBP", "JPY", "CHF"]
PURPOSE_CODES    = ["SALA", "SUPP", "TRAD", "LOAN", "INVS"]
NORMAL_NAMES     = ["Alice Johnson", "James Smith", "Maria Santos",
                    "Wei Zhang", "Priya Patel", "Carlos Rivera",
                    "Emma Wilson", "Liam Brown", "Yuki Tanaka"]

def make_transaction(i):
    is_suspicious = (i % 20 == 0)          # Every 20th is bad (~5%)
    txn_type = random.choice(
        ["sanctioned", "missing_fields", "large_amount"]
    ) if is_suspicious else "normal"

    return {
        "transaction_id":      str(uuid.uuid4()),
        "timestamp":           datetime.now(timezone.utc).isoformat(),
        "message_type":        "pacs.008",
        "sender_name":         random.choice(SANCTIONED_NAMES)
                               if txn_type == "sanctioned"
                               else random.choice(NORMAL_NAMES),
        "sender_account":      f"DE{''.join([str(random.randint(0,9)) for _ in range(18)])}",
        "sender_country":      random.choice(HIGH_RISK)
                               if txn_type == "large_amount"
                               else random.choice(NORMAL_COUNTRIES),
        "sender_address":      "" if txn_type == "missing_fields"
                               else f"{random.randint(1,999)} Main St, City",
        "receiver_name":       random.choice(NORMAL_NAMES),
        "receiver_account":    f"GB{''.join([str(random.randint(0,9)) for _ in range(18)])}",
        "receiver_country":    random.choice(NORMAL_COUNTRIES),
        "amount":              round(random.uniform(500_000, 5_000_000), 2)
                               if txn_type == "large_amount"
                               else round(random.uniform(100, 50_000), 2),
        "currency":            random.choice(CURRENCIES),
        "purpose_code":        random.choice(PURPOSE_CODES),
        "transaction_type":    txn_type,
    }

# -- Generate 100 transactions --------------------------------
transactions = [make_transaction(i) for i in range(1, 101)]

# -- Write to Unity Catalog Volume ----------------------------
volume_path = "/Volumes/aml_pipeline/transactions/raw_data/batch_001.json"

with open(volume_path, "w") as f:
    for txn in transactions:
        f.write(json.dumps(txn) + "\n")   # One JSON object per line

print(f"✅ Written 100 transactions to Volume!")
print(f"📄 File: {volume_path}")
print(f"\nSample transaction:")
print(json.dumps(transactions[0], indent=2))
print(f"\n🚨 Sample suspicious transaction:")
sus = next(t for t in transactions if t["transaction_type"] != "normal")
print(json.dumps(sus, indent=2))

✅ Written 100 transactions to Volume!
📄 File: /Volumes/aml_pipeline/transactions/raw_data/batch_001.json

Sample transaction:
{
  "transaction_id": "d13249cd-fd1c-4314-9874-b2859cafcb07",
  "timestamp": "2026-05-10T00:47:22.160381+00:00",
  "message_type": "pacs.008",
  "sender_name": "Maria Santos",
  "sender_account": "DE566551377601453261",
  "sender_country": "DE",
  "sender_address": "845 Main St, City",
  "receiver_name": "James Smith",
  "receiver_account": "GB096513774846474949",
  "receiver_country": "DE",
  "amount": 34297.53,
  "currency": "JPY",
  "purpose_code": "SUPP",
  "transaction_type": "normal"
}

🚨 Sample suspicious transaction:
{
  "transaction_id": "6aa2955e-085b-442c-bedb-4bc81850b5bd",
  "timestamp": "2026-05-10T00:47:22.161232+00:00",
  "message_type": "pacs.008",
  "sender_name": "Liam Brown",
  "sender_account": "DE915273116034072391",
  "sender_country": "IR",
  "sender_address": "477 Main St, City",
  "receiver_name": "Emma Wilson",
  "receiver_account": "G

In [0]:
# ============================================================
# CELL 3 — Auto Loader: Stream JSON files → Bronze Delta Table
# ============================================================
# Auto Loader is Databricks' tool for reading files as a stream.
# It watches a folder, and every time a new file appears,
# it automatically reads it and adds it to your Delta table.
# This is exactly how real banks ingest transaction files.

# -- Paths ---------------------------------------------------
RAW_PATH        = "/Volumes/aml_pipeline/transactions/raw_data/"
BRONZE_TABLE    = "aml_pipeline.transactions.bronze_transactions"
CHECKPOINT_PATH = "/Volumes/aml_pipeline/transactions/raw_data/_checkpoints/bronze"

# -- Define the exact schema of our JSON transactions --------
# Telling Spark exactly what fields to expect makes it
# faster and more reliable than letting it guess.
from pyspark.sql.functions import current_timestamp, col, lit
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)

schema = StructType([
    StructField("transaction_id",      StringType(),    True),
    StructField("timestamp",           StringType(),    True),
    StructField("message_type",        StringType(),    True),
    StructField("sender_name",         StringType(),    True),
    StructField("sender_account",      StringType(),    True),
    StructField("sender_country",      StringType(),    True),
    StructField("sender_address",      StringType(),    True),
    StructField("receiver_name",       StringType(),    True),
    StructField("receiver_account",    StringType(),    True),
    StructField("receiver_country",    StringType(),    True),
    StructField("amount",              DoubleType(),    True),
    StructField("currency",            StringType(),    True),
    StructField("purpose_code",        StringType(),    True),
    StructField("transaction_type",    StringType(),    True),
])

# -- Auto Loader reads the raw JSON files as a stream --------
bronze_stream = (
    spark.readStream
         .format("cloudFiles")            # Auto Loader format
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "/schema")
         .schema(schema)
         .load(RAW_PATH)
)

# -- Add ingestion metadata columns --------------------------
# These are extra columns we add to track WHEN and HOW
# each record entered our pipeline — critical for audit trails
from pyspark.sql.functions import current_timestamp, input_file_name, lit

bronze_with_metadata = (
    bronze_stream
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file",         col("_metadata.file_path"))
    .withColumn("pipeline_layer",      lit("bronze"))
)

# -- Write stream to Bronze Delta Table ----------------------
bronze_query = (
    bronze_with_metadata
    .writeStream
    .format("delta")                      # Save as Delta Lake table
    .outputMode("append")                 # Add new records, never delete
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)           # Process all available files now
    .toTable(BRONZE_TABLE)                # Save directly to Unity Catalog
)

bronze_query.awaitTermination()
print("✅ Bronze layer ingestion complete!")

✅ Bronze layer ingestion complete!


In [0]:
# ============================================================
# CELL 4 — Verify: Check what landed in the Bronze table
# ============================================================

# -- How many records came in? -------------------------------
count = spark.table("aml_pipeline.transactions.bronze_transactions").count()
print(f"✅ Total records in Bronze table: {count}")

# -- Show the first 5 records --------------------------------
print("\n📋 Sample records:")
spark.table("aml_pipeline.transactions.bronze_transactions") \
     .select("transaction_id", "sender_name", "sender_country",
             "amount", "currency", "transaction_type",
             "ingestion_timestamp") \
     .show(5, truncate=False)

# -- How many suspicious vs normal? --------------------------
print("🔍 Transaction type breakdown:")
spark.table("aml_pipeline.transactions.bronze_transactions") \
     .groupBy("transaction_type") \
     .count() \
     .orderBy("count", ascending=False) \
     .show()

# -- Check our audit columns are there -----------------------
print("🏷️  Audit columns check:")
spark.table("aml_pipeline.transactions.bronze_transactions") \
     .select("ingestion_timestamp", "source_file", "pipeline_layer") \
     .show(3, truncate=False)

✅ Total records in Bronze table: 100

📋 Sample records:
+------------------------------------+-------------+--------------+--------+--------+----------------+-----------------------+
|transaction_id                      |sender_name  |sender_country|amount  |currency|transaction_type|ingestion_timestamp    |
+------------------------------------+-------------+--------------+--------+--------+----------------+-----------------------+
|d13249cd-fd1c-4314-9874-b2859cafcb07|Maria Santos |DE            |34297.53|JPY     |normal          |2026-05-10 03:02:50.174|
|b789819a-c1a3-4d15-a910-0af629655987|Alice Johnson|DE            |23784.45|GBP     |normal          |2026-05-10 03:02:50.174|
|8aff7ee2-079c-448f-abd7-5da7e2cf17c1|Priya Patel  |CA            |39153.52|EUR     |normal          |2026-05-10 03:02:50.174|
|12d8313f-b70b-4734-8f24-b8d00c58b92d|Alice Johnson|GB            |25989.69|EUR     |normal          |2026-05-10 03:02:50.174|
|77ef4f59-d5d8-44cf-bfe6-b3bc619764ed|James Smith  |CA 